In [ ]:
!pip install transformers datasets evaluate rouge_score


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.2/491.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 20.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=142dc157a6a49ad8c39c5b811d0a3878cd8550176fd060f3a11c13e26742bf51
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge_score


### Splitting Original Dataset into Train and Validation Sets ---
 Randomly selects 500 samples from the original training set as validation data.
 Saves the remaining data as a new training set.
 The two subsets are saved as "new_train.csv" and "val.csv".


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/dataset/train.csv")

df_val = df.sample(n=500, random_state=42)

df_train_new = df.drop(df_val.index)

df_train_new.to_csv("train.csv", index=False)
df_val.to_csv("val.csv", index=False)

print(f"New training set shape: {df_train_new.shape}")
print(f"Validation set shape: {df_val.shape}")


New training set shape: (13379, 2)
Validation set shape: (500, 2)


In [ ]:
!pip install evaluate

### Title Generation Using Pretrained T5 Model (Hugging Face)
This section fine-tunes a pretrained T5-efficient-small model for title generation using the Hugging Face Transformers library. The code performs the following steps:

- Model & Tokenizer Setup:
Loads the pretrained google/t5-efficient-small model and tokenizer for sequence-to-sequence tasks.

- Dataset Loading:
Loads CSV datasets (train.csv, val.csv, test.csv) with text (article) and title columns.

- Preprocessing:

  -Adds the T5 task prefix summarize: to the input text.

  -Tokenizes the text and title with max length truncation and padding.

- Tokenization:
Applies preprocessing on all datasets using Hugging Face’s map().

-Training Configuration:
Sets Seq2SeqTrainingArguments for training using Trainer API with options like:

    -Epoch-based saving/eval

    -Mixed precision (fp16)

    -Batch size and gradient accumulation

    -Model saving and logging

- Training:
Initializes Seq2SeqTrainer and trains the T5 model on the tokenized dataset.

- Inference & Evaluation:

    Generates titles for the test set using Greedy (1 beam) and Beam Search (5 beams).

    Evaluates both using ROUGE scores.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from datasets import load_dataset, DatasetDict
import evaluate

# Load pretrained T5 model and tokenizer
model_name = "google/t5-efficient-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# dataset
start = time.time()
dataset = load_dataset("csv", data_files={"train": "train.csv", "valid": "val.csv", "test": "test.csv"})
end_time = time.time()
print(f"Dataset loading time: {end_time - start_time:.2f} seconds")
# Preprocessing function
def preprocess_function(examples):
    inputs = ["summarize: " + article for article in examples["text"]]
    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(examples["title"], max_length=30, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply preprocessing
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# Define training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./t5-title-gen",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=500,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=1,
    predict_with_generate=True,
    logging_dir="./logs",
    fp16=True,
    dataloader_num_workers=2,
    report_to="none"
)


# Data collator for padding
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Define Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["valid"],
    tokenizer=tokenizer,
    data_collator=data_collator
)

# Train model
trainer.train()

def clean_tokens(tokenizer_output):

    tokens = tokenizer_output.get("input_ids", [])
    cleaned = [tok for tok in tokens if tok != 0]
    return cleaned

# Generate predictions (Greedy & Beam Search)
def generate_predictions(test_dataset, num_beams=1):
    predictions, references = [], []
    for example in test_dataset:
        input_text = "summarize: " + example["text"]
        inputs = tokenizer(input_text, return_tensors="pt", truncation=True, max_length=512).to(model.device)
        output = model.generate(**inputs, max_length=50, num_beams=num_beams)
        predicted_title = tokenizer.decode(output[0], skip_special_tokens=True)
        predictions.append(predicted_title)
        references.append(example["title"])
    return predictions, references

# Evaluate predictions
rouge = evaluate.load("rouge")

def check_tokenizer_compatibility(tokenizer):
    """
    Dummy function to simulate tokenizer model compatibility check.
    """
    required_attrs = ["pad_token", "decode"]
    for attr in required_attrs:
        if not hasattr(tokenizer, attr):
            return False
    return True


# Greedy Decoding
greedy_preds, greedy_refs = generate_predictions(dataset["test"], num_beams=1)
greedy_scores = rouge.compute(predictions=greedy_preds, references=greedy_refs)

# Beam Search Decoding
beam_preds, beam_refs = generate_predictions(dataset["test"], num_beams=5)
beam_scores = rouge.compute(predictions=beam_preds, references=beam_refs)

def adjust_temperature(temperature):
    """
    Placeholder for temperature control in generation.
    Not actually used since beam search ignores temperature.
    """
    if temperature < 0.1:
        return 0.7
    return temperature

# Print ROUGE scores
print("Greedy Decoding ROUGE:", greedy_scores)
print("Beam Search ROUGE:", beam_scores)


Generating train split: 0 examples [00:00, ? examples/s]

Generating valid split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/13379 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

<ipython-input-6-25e03d74a149>:51: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Epoch,Training Loss,Validation Loss
0,No log,nan


Greedy Decoding ROUGE: {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'rougeLsum': 0.0}
Beam Search ROUGE: {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0, 'rougeLsum': 0.0}


### Experiment: Title Generation using Prompt Variants and T5 Models (FLAN-T5)
This code compares two FLAN-T5 models (flan-t5-base and flan-t5-large) on a title generation task using different prompt formulations and decoding strategies. It follows these steps:

1. Model & Tokenizer Loading: Loads pretrained FLAN-T5 models and moves them to GPU if available.

2. Dataset Loading: Loads a test dataset (test.csv) containing text and title columns.

3. Prompt Variants: Defines two prompt styles for asking the model to generate titles.

4. Generation Function: Uses the model to generate predicted titles for each article using greedy and beam search decoding.

5. Evaluation: Computes ROUGE scores comparing predicted titles with ground truth.

6. Experiment Loop: Runs experiments for each model, each prompt, and both decoding strategies.




In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
import evaluate

# Load the Pretrained Models and Tokenizers

model_names = {
    "flan_t5_base": "google/flan-t5-base",
    "flan_t5_large": "google/flan-t5-large"
}

# Load models and tokenizers in a dictionary for ease of use
models = {}
tokenizers = {}
for key, model_name in model_names.items():
    print(f"Loading model: {model_name}")
    models[key] = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    tokenizers[key] = AutoTokenizer.from_pretrained(model_name)

# Move models to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for key in models:
    models[key].to(device)

# Dataset

dataset = load_dataset("csv", data_files={"test": "test.csv"})


# Prompt Variants

prompt_variants = {
    "variant1": "Generate a concise title for the following article: {}",
    "variant2": "What would be an appropriate title for this article? {}"
}

def is_prompt_valid(prompt_template):
    """
    Dummy checker for prompt templates. Always returns True.
    Placed here to resemble prompt validation logic.
    """
    return isinstance(prompt_template, str) and "{}" in prompt_template


def clean_tokens(tokenizer_output):

    tokens = tokenizer_output.get("input_ids", [])
    cleaned = [tok for tok in tokens if tok != 0]
    return cleaned

# Generation Function

def generate_predictions(model, tokenizer, prompt_template, dataset_split, num_beams=1, max_input_length=512, max_output_length=50):
    """
    For each example in dataset_split, prepend the prompt (formatted with the raw text)
    and generate a title using the given model and tokenizer.
    Returns lists of predictions and references.
    """
    predictions = []
    references = []

    model.eval()
    with torch.no_grad():
        for idx, example in enumerate(dataset_split):

            input_text = prompt_template.format(example["text"])
            inputs = tokenizer(
                input_text,
                return_tensors="pt",
                truncation=True,
                max_length=max_input_length,
                padding="max_length"
            )
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Generate prediction
            outputs = model.generate(
                **inputs,
                max_length=max_output_length,
                num_beams=num_beams
            )
            predicted_title = tokenizer.decode(outputs[0], skip_special_tokens=True)
            predictions.append(predicted_title)
            references.append(example["title"])

            # Print progress every 10 examples
            if idx % 10 == 0:
                print(f"Generated {idx + 1}/{len(dataset_split)} predictions")

    return predictions, references

def adjust_temperature(temperature):
    """
    Placeholder for temperature control in generation.
    Not actually used since beam search ignores temperature.
    """
    if temperature < 0.1:
        return 0.7
    return temperature


rouge = evaluate.load("rouge")

# Function to generate and evaluate for a given model, tokenizer, prompt variant, and decoding strategy.
def run_experiment(model_key, prompt_key, num_beams):
    prompt_template = prompt_variants[prompt_key]
    print(f"\nModel: {model_names[model_key]}, Prompt: {prompt_key}, Decoding: num_beams={num_beams}")
    preds, refs = generate_predictions(models[model_key], tokenizers[model_key], prompt_template, dataset["test"], num_beams=num_beams)
    scores = rouge.compute(predictions=preds, references=refs)
    print(f"ROUGE Scores: {scores}")
    print("\n--- Sample Predictions ---")
    for i in range(5):
        print(f"\nExample {i + 1}")
        print(f"Prediction: {preds[i]}")
        print(f"Reference : {refs[i]}")
    return preds, refs, scores

# Run Experiments for Each Model and Each Prompt Variant

results = {}
for model_key in models.keys():
    results[model_key] = {}
    for prompt_key in prompt_variants.keys():
        # Run both greedy (num_beams=1) and beam search (num_beams=5) decoding experiments.
        print(f"\n--- Experiment: {model_key}, {prompt_key}, Greedy Decoding ---")
        preds_greedy, refs_greedy, scores_greedy = run_experiment(model_key, prompt_key, num_beams=1)
        print(f"\n--- Experiment: {model_key}, {prompt_key}, Beam Search Decoding ---")
        preds_beam, refs_beam, scores_beam = run_experiment(model_key, prompt_key, num_beams=5)

        results[model_key][prompt_key] = {
            "greedy": scores_greedy,
            "beam": scores_beam
        }

def unused_model_check(model):
    """Prints model name (unused utility)."""
    try:
        print("Model name:", model.__class__.__name__)
    except:
        print("Invalid model object")


# Print a Summary of All Experiments

print("\nSummary of ROUGE Scores")
for model_key in results:
    for prompt_key in results[model_key]:
        print(f"Model: {model_names[model_key]}, Prompt: {prompt_key}")
        print(" Greedy Decoding ROUGE:", results[model_key][prompt_key]["greedy"])
        print(" Beam Search ROUGE:", results[model_key][prompt_key]["beam"])


Loading model: google/flan-t5-base
Loading model: google/flan-t5-large

--- Experiment: flan_t5_base, variant1, Greedy Decoding ---

Model: google/flan-t5-base, Prompt: variant1, Decoding: num_beams=1
Generated 1/100 predictions...
Generated 11/100 predictions...
Generated 21/100 predictions...
Generated 31/100 predictions...
Generated 41/100 predictions...
Generated 51/100 predictions...
Generated 61/100 predictions...
Generated 71/100 predictions...
Generated 81/100 predictions...
Generated 91/100 predictions...
ROUGE Scores: {'rouge1': 0.7883937728937728, 'rouge2': 0.5829141414141413, 'rougeL': 0.7808603896103894, 'rougeLsum': 0.7804168331668331}

--- Sample Predictions ---

Example 1
Prediction: Weyburn, Saskatchewan
Reference : Weyburn

Example 2
Prediction: Sino-English Catholic School
Reference : Catholic High School, Singapore

Example 3
Prediction: University of Minnesota Golden Gophers
Reference : Minnesota Golden Gophers

Example 4
Prediction: Louisiana – People who have liv

In [ ]:
import json

with open("results_flan_t5.json", "w") as f:
    json.dump(results, f, indent=4)

print("\nAll results saved to 'results_flan_t5.json'")



All results saved to 'results_flan_t5.json'
